# 11 - Image : Modèles Avancés et Ensembles

## Objectif de ce notebook
Explorer les **ensembles de modèles** (Voting, Stacking) et l'**interprétabilité** pour finaliser la chaîne de classification par images.

**Prérequis** : Exécuter les notebooks 09 et 10 pour disposer des features et du meilleur modèle optimisé.

**Livrable** : Modèle final image retenu, comparaison avec le pipeline texte, conclusions.

---

## Plan
1. Chargement des données et des modèles (notebook 10)
2. Ensembles avancés (Voting, Stacking)
3. Comparaison finale Texte vs Image et conclusions

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count() or 4)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

sys.path.append(str(Path('../').resolve()))

from src.evaluation import evaluate_model, plot_confusion_matrix

from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliothèques importées")

In [ ]:
ROOT = Path('../').resolve()
MODELS_DIR = ROOT / 'models'
cache = np.load(MODELS_DIR / 'image_features_cache.npz', allow_pickle=True)
X_features = cache['X']
y_labels = cache['y']

# Superclasse (24 classes) : Publications -> 9999
CODE_SUPERCLASSE = 9999
PUBLICATIONS_CLASSES = {10, 2280, 2403, 2705}

y_labels_superclass = np.array([
    CODE_SUPERCLASSE if y in PUBLICATIONS_CLASSES else y
    for y in y_labels
])

with open(MODELS_DIR / 'label_encoder_image.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

label_encoder_superclass_path = MODELS_DIR / 'label_encoder_image_superclass.pkl'
if label_encoder_superclass_path.exists():
    with open(label_encoder_superclass_path, 'rb') as f:
        label_encoder_superclass = pickle.load(f)
else:
    label_encoder_superclass = LabelEncoder()
    label_encoder_superclass.fit(y_labels_superclass)

# Encodage labels
y_encoded = label_encoder.transform(y_labels)
y_encoded_sc = label_encoder_superclass.transform(y_labels_superclass)

SCENARIOS = [
    ("27 classes", y_encoded, label_encoder),
    ("24 classes (superclass)", y_encoded_sc, label_encoder_superclass)
]

indices = np.arange(len(y_labels))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y_encoded
)

X_train_split = X_features[train_idx]
X_val_split = X_features[val_idx]

splits_by_scenario = {name: (y_enc[train_idx], y_enc[val_idx]) for name, y_enc, _ in SCENARIOS}

print(f"✅ Données : {X_train_split.shape[0]:,} train | {X_val_split.shape[0]:,} val")
print(f"✅ Scénarios : {[s[0] for s in SCENARIOS]}")

## 2. Ensembles (Voting, Stacking)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)),
    ('svm', LinearSVC(max_iter=2000, random_state=42, dual=False)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)),
]

metrics_voting_by_scenario = {}
metrics_stacking_by_scenario = {}

for scenario_name, _, _ in SCENARIOS:
    y_train_s, y_val_s = splits_by_scenario[scenario_name]

    voting = VotingClassifier(estimators, voting='hard')
    voting.fit(X_train_split, y_train_s)
    y_pred_voting = voting.predict(X_val_split)
    m_voting = evaluate_model(y_val_s, y_pred_voting)
    metrics_voting_by_scenario[scenario_name] = m_voting
    print(f"VotingClassifier ({scenario_name}) F1-macro : {m_voting['f1_macro']:.4f}")

    stacking = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=1000, n_jobs=-1),
        cv=3
    )
    stacking.fit(X_train_split, y_train_s)
    y_pred_stack = stacking.predict(X_val_split)
    m_stack = evaluate_model(y_val_s, y_pred_stack)
    metrics_stacking_by_scenario[scenario_name] = m_stack
    print(f"StackingClassifier ({scenario_name}) F1-macro : {m_stack['f1_macro']:.4f}")

## 3. Conclusions et comparaison Texte vs Image

In [ ]:
text_results = MODELS_DIR / 'model_comparison_results.csv'
image_results = MODELS_DIR / 'image_baseline_comparison.csv'

if text_results.exists() and image_results.exists():
    df_text = pd.read_csv(text_results)
    df_image = pd.read_csv(image_results)

    print("📊 Comparaison Texte vs Image (meilleurs F1-macro) :")

    if 'Scenario' in df_text.columns:
        for scenario in df_text['Scenario'].unique():
            best_text = df_text[df_text['Scenario'] == scenario]['F1_macro'].max()
            print(f"   Texte best ({scenario}) : {best_text:.4f}")
    else:
        print(f"   Texte best : {df_text['F1_macro'].max():.4f}")

    if 'Scenario' in df_image.columns:
        for scenario in df_image['Scenario'].unique():
            best_image = df_image[df_image['Scenario'] == scenario]['F1_macro'].max()
            print(f"   Image best ({scenario}) : {best_image:.4f}")
    else:
        print(f"   Image best : {df_image['F1_macro'].max():.4f}")
else:
    print("Exécuter les notebooks texte et image pour la comparaison.")

print("\n✅ Pipeline image terminé.")